# Food Delivery Time Prediction Using Machine Learning Algorithms
**Machine Learning Mini Project | TY Computer Engineering**

This notebook compares five regression algorithms to predict delivery time in minutes.

**Algorithms:** Linear Regression, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor, and Support Vector Regression (SVR).

**Important:** Use a legitimate dataset and document its source. The notebook supports a CSV upload. If you do not yet have a dataset, it creates a clearly labelled *synthetic demonstration dataset* so you can test the workflow; synthetic results must not be presented as results from real deliveries.

## 1. Import libraries

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load the dataset
Upload a CSV file if you have one. Otherwise, run the next cell to create a synthetic dataset for checking that the code works. Replace it with a real public dataset before final submission and update the dataset citation in your report.

In [ ]:
# Google Colab upload. If no file is uploaded, the notebook will offer a synthetic demo dataset.
USE_SYNTHETIC_DEMO_IF_NO_UPLOAD = True

try:
    from google.colab import files
    uploaded = files.upload()
    csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
except Exception:
    csv_files = []

if csv_files:
    DATA_PATH = csv_files[0]
    df = pd.read_csv(DATA_PATH)
    DATA_SOURCE_NOTE = f"User-uploaded dataset: {DATA_PATH}. Record the original source/license in your report."
    print("Loaded:", DATA_PATH)
else:
    if not USE_SYNTHETIC_DEMO_IF_NO_UPLOAD:
        raise FileNotFoundError("Upload a CSV dataset to continue.")
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1200
    distance = rng.uniform(0.5, 15, n)
    prep = rng.integers(5, 26, n)
    traffic = rng.choice(["Low", "Medium", "High"], n, p=[0.3, 0.45, 0.25])
    weather = rng.choice(["Clear", "Cloudy", "Rainy"], n, p=[0.55, 0.25, 0.20])
    vehicle = rng.choice(["Bike", "Scooter"], n)
    experience = rng.integers(0, 11, n)
    hour = rng.integers(10, 23, n)
    traffic_delay = pd.Series(traffic).map({"Low": 2, "Medium": 7, "High": 14}).to_numpy()
    weather_delay = pd.Series(weather).map({"Clear": 0, "Cloudy": 2, "Rainy": 7}).to_numpy()
    noise = rng.normal(0, 3.5, n)
    duration = 8 + 2.2*distance + 0.8*prep + traffic_delay + weather_delay - 0.35*experience + np.where((hour >= 19) & (hour <= 21), 4, 0) + noise
    duration = np.clip(duration, 8, None).round(1)
    df = pd.DataFrame({
        "Distance_km": distance.round(2),
        "Preparation_Time_min": prep,
        "Traffic_Level": traffic,
        "Weather": weather,
        "Vehicle_Type": vehicle,
        "Rider_Experience_years": experience,
        "Order_Hour": hour,
        "Delivery_Time_min": duration
    })
    DATA_SOURCE_NOTE = "SYNTHETIC DEMONSTRATION DATA generated in notebook. Not real-world evidence; replace with a public dataset before final submission."
    print("No CSV uploaded. Created synthetic demonstration data only.")
    print("For final submission, upload your real dataset and update DATA_SOURCE_NOTE.")
df.head()

## 3. Inspect and clean the data
Set `TARGET_COLUMN` to the exact column name containing actual delivery duration in minutes. Update `DROP_COLUMNS` to remove IDs or columns that would leak information unavailable at the moment the prediction is made.

For prediction at order placement, avoid features such as actual pickup time or actual trip duration components if they are only known later.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
display(df.head())
display(df.isna().sum().to_frame("missing_values"))

TARGET_COLUMN = "Delivery_Time_min"  # CHANGE this to match your dataset exactly
DROP_COLUMNS = []  # Example: ["ID", "Delivery_ID"]

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' not found. Available columns: {df.columns.tolist()}. "
        "Edit TARGET_COLUMN to match your dataset."
    )

df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns]).copy()
df = df.dropna(subset=[TARGET_COLUMN])
df[TARGET_COLUMN] = pd.to_numeric(df[TARGET_COLUMN], errors="coerce")
df = df.dropna(subset=[TARGET_COLUMN])

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Remove columns with a single unique value; they provide no predictive information.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Removing constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

if X.shape[1] == 0:
    raise ValueError("No feature columns remain after removing target/constant columns.")
print("Feature columns:", X.columns.tolist())
print("Target:", TARGET_COLUMN)
print("Rows after basic cleaning:", len(df))

## 4. Exploratory data analysis

In [ ]:
plt.figure(figsize=(8, 4))
y.hist(bins=30)
plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery time (minutes)")
plt.ylabel("Number of orders")
plt.tight_layout()
plt.show()

numeric_for_corr = df.select_dtypes(include=np.number)
if numeric_for_corr.shape[1] >= 2:
    plt.figure(figsize=(9, 6))
    plt.imshow(numeric_for_corr.corr(numeric_only=True), aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns, rotation=75, ha="right")
    plt.yticks(range(len(numeric_for_corr.columns)), numeric_for_corr.columns)
    plt.title("Numeric Feature Correlation")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric columns for a correlation plot.")

## 5. Preprocessing and train/test split
We split the data before fitting preprocessing steps to avoid learning imputation, scaling or category encodings from the test set. Numeric features are median-imputed; categorical features are most-frequent-imputed and one-hot encoded. SVR and Linear Regression use scaled numeric features.

In [ ]:
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_unscaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot)
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("num", numeric_scaled, numeric_features),
    ("cat", categorical_pipe, categorical_features)
], remainder="drop")

preprocessor_tree = ColumnTransformer(transformers=[
    ("num", numeric_unscaled, numeric_features),
    ("cat", categorical_pipe, categorical_features)
], remainder="drop")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

## 6. Train five regression algorithms

In [ ]:
models = {
    "Linear Regression": Pipeline([
        ("preprocess", preprocessor_scaled),
        ("model", LinearRegression())
    ]),
    "Decision Tree Regressor": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE))
    ]),
    "Random Forest Regressor": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", RandomForestRegressor(n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2))
    ]),
    "Gradient Boosting Regressor": Pipeline([
        ("preprocess", preprocessor_tree),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE, n_estimators=150, learning_rate=0.05, max_depth=2))
    ]),
    "Support Vector Regression (SVR)": Pipeline([
        ("preprocess", preprocessor_scaled),
        ("model", SVR(C=10, epsilon=2.0, kernel="rbf"))
    ])
}

results = []
fitted_models = {}
predictions = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)
    fitted_models[name] = pipeline
    predictions[name] = pred
    results.append({
        "Algorithm": name,
        "MAE (minutes)": mean_absolute_error(y_test, pred),
        "RMSE (minutes)": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 Score": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("MAE (minutes)").reset_index(drop=True)
display(results_df.round(3))

## 7. Compare model performance

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["MAE (minutes)"])
plt.title("Model Comparison — Mean Absolute Error (lower is better)")
plt.ylabel("MAE (minutes)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["RMSE (minutes)"])
plt.title("Model Comparison — RMSE (lower is better)")
plt.ylabel("RMSE (minutes)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

best_name = results_df.iloc[0]["Algorithm"]
best_model = fitted_models[best_name]
print("Best model by MAE on this test split:", best_name)
print("Use this conclusion only after running the notebook on your final dataset.")

## 8. Actual vs predicted for the best model

In [ ]:
best_pred = predictions[best_name]
plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_pred, alpha=0.65)
lims = [min(float(y_test.min()), float(np.min(best_pred))), max(float(y_test.max()), float(np.max(best_pred)))]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual delivery time (minutes)")
plt.ylabel("Predicted delivery time (minutes)")
plt.title(f"Actual vs Predicted — {best_name}")
plt.tight_layout()
plt.show()

error_df = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": best_pred})
error_df["Absolute_Error"] = np.abs(error_df["Actual"] - error_df["Predicted"])
display(error_df.head(10).round(2))

## 9. Save the best model and results

In [ ]:
os.makedirs("outputs", exist_ok=True)
joblib.dump(best_model, "outputs/best_delivery_time_model.joblib")
results_df.to_csv("outputs/model_comparison_results.csv", index=False)
error_df.to_csv("outputs/test_predictions.csv", index=False)

summary = {
    "best_model_by_mae": best_name,
    "target_column": TARGET_COLUMN,
    "dataset_source_note": DATA_SOURCE_NOTE,
    "rows_used": int(len(df)),
    "features_used": X.columns.tolist(),
    "test_mae_minutes": float(results_df.iloc[0]["MAE (minutes)"]),
    "test_rmse_minutes": float(results_df.iloc[0]["RMSE (minutes)"]),
    "test_r2": float(results_df.iloc[0]["R2 Score"])
}
with open("outputs/run_summary.json", "w", encoding="utf-8") as f:
    import json
    json.dump(summary, f, indent=2)

print("Saved model and result files under outputs/.")
print(json.dumps(summary, indent=2))

## 10. Try one prediction
Edit the example below so every input column matches the feature columns in your chosen dataset. This uses a row from the held-out test set, so it is a safe way to confirm the saved pipeline predicts in the expected units.

For a custom input interface, the Streamlit app must use the same feature names and data types as the trained model.

In [ ]:
example_input = X_test.iloc[[0]].copy()
example_prediction = best_model.predict(example_input)[0]
print("Example input:")
display(example_input)
print(f"Predicted delivery time: {example_prediction:.1f} minutes")
print(f"Actual delivery time: {float(y_test.iloc[0]):.1f} minutes")

## 11. Report notes — fill these after execution
- Dataset name, source URL, license and number of usable rows:
- Target variable and feature list:
- Train/test split:
- Best model by MAE:
- MAE, RMSE and R² for all five models:
- Limitations and possible future improvements:

**Academic integrity:** Do not report synthetic demonstration results as real-world results. Include the true dataset source and report the actual outputs from your final run.